In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

# add gsl includes to root
environ["ROOT_INCLUDE_PATH"] = environ["ROOT_INCLUDE_PATH"] + ":" + environ["GSL_ROOT_DIR"] + "/include"

In [2]:
import ROOT
from analysis_framework import Dataset, Analysis

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x88cd2d0


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 6
# prod = False
prod = True
no_rvec = True
write_outputs = False
# write_outputs = True
# plot_dir_postfix = "-new-cuts"
dataset_path = "data/datasets/selected-objects/test.json"
# output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/reweighted/test"
# output_meta_path = "data/datasets/reweighted"
# output_meta = f"{output_meta_path}/test.json"
# checked_output_meta = f"{output_meta_path}/checked-test.json"
# output_collections = [
#     "true_lep_lvec", "true_nu_lvec", "true_quark1_lvec", "true_quark2_lvec",
#     "iso_lep_lvec", "nu_lvec", "R2Jet_sel1_lvec", "R2Jet_sel2_lvec",
#     ]
# true lvecs do not exist in every df so cannot be explicitly requested...
# urgh but empty snapshots are also not allowed
# output_collections = r"(true_\w+_lvec)|(iso_lep_lvec)|(nu_lvec)|(R2Jet_sel1_lvec)|(R2Jet_sel2_lvec)|(\w*sqme\w*)|(weight\w*)"
# plot_dir = f"plots/pre-selection/test{plot_dir_postfix}"
if prod:
    # dataset_path = "data/datasets/miniDSTs/processed-no-exc-higgs.json"
    # dataset_path = "data/datasets/miniDSTs/processed-no-exc-higgs-min-aa-min-higgs.json"
    # dataset_path = "data/datasets/miniDSTs/min-higgs.json"
    dataset_path = "data/datasets/selected-objects/signal-only.json"
    # output_path = "root://eospublic.cern.ch//eos/experiment/clicdp/data/user/l/lreichen/snapshots3/min-higgs-d"
    # output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/reweighted/signal-only"
    # output_meta_path = "data/datasets/reweighted"
    # output_meta = f"{output_meta_path}/signal-only.json"
    # checked_output_meta = f"{output_meta_path}/checked-signal-only.json"
    # plot_dir = "plots/pre-selection/full"
    # plot_dir = f"plots/pre-selection/min-higgs{plot_dir_postfix}"


In [4]:
# ROOT.EnableImplicitMT(n_threads)
# environ["OMP_NUM_THREADS"] = "6"

In [5]:
dataset = Dataset.from_json(dataset_path)

In [6]:
analysis = Analysis(dataset)

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xbadebb0


In [7]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]

True


In [8]:
ROOT.gInterpreter.Declare("#include \"kinfit.h\"")
ROOT.gSystem.Load("libMarlinKinfit.so")
# E_err = 4.4
E_err = 3.5
Theta_err = 0.045
Phi_err = 0.04
E_cms = 250.
m_W = 80.419
width_W = 2.049
fitter = ROOT.enuWFit(E_err, Theta_err, Phi_err, x_angle, E_cms, m_W, width_W)

In [9]:
analysis.Define("fitres", fitter, ["iso_lep_lvec", "nu_lvec", "R2Jet1_lvec", "R2Jet2_lvec"])
analysis.Define("prob", "fitres.prob")
analysis.Define("chi2", "fitres.chi2")
analysis.Define("error", "fitres.error")

In [10]:
analysis.book_histogram_1D("prob", "prob", ("", "", 100, 0., 1.))
analysis.book_histogram_1D("chi2", "chi2", ("", "", 100, 0., 50.))
analysis.book_histogram_1D("error", "error", ("", "", 20, -10., 10.))

In [11]:
if write_outputs:
    analysis.book_snapshots("events", output_path, output_meta, output_collections, no_rvec=no_rvec)

In [12]:
%%time
analysis.run()

CPU times: user 36.3 s, sys: 347 ms, total: 36.7 s
Wall time: 41.5 s


In [13]:
# if write_outputs:
    # analysis.check_snapshots("events", output_path, checked_output_meta)

In [14]:
analysis.draw_histogram("prob")
analysis.draw_histogram("chi2")
analysis.draw_histogram("error")

(<cppyy.gbl.THStack object at 0x101357b0>,
 <cppyy.gbl.TCanvas object at 0xee69bc0>)